What does the movie lense dataset look like?

In [4]:
import torch
import torch_geometric as tg

In [5]:
dataset = tg.datasets.MovieLens("/Users/emilie/data/movie_lens/")

Processing...


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Batches:   0%|          | 0/305 [00:00<?, ?it/s]

Done!


In [14]:
len(dataset)

1

In [20]:
dataset.num_node_features, dataset.num_edge_features

({'movie': 404, 'user': 0}, {('user', 'rates', 'movie'): 0})

In [17]:
graph = dataset[0]

In [18]:
graph

HeteroData(
  movie={ x=[9742, 404] },
  user={ num_nodes=610 },
  (user, rates, movie)={
    edge_index=[2, 100836],
    edge_label=[100836],
    time=[100836],
  }
)

In [41]:
graph['movie'].x.shape

torch.Size([9742, 404])

In [47]:
graph['movie'].x[:10]

tensor([[-0.0828,  0.0530,  0.0536,  ...,  0.0000,  0.0000,  0.0000],
        [-0.1053,  0.1508, -0.0264,  ...,  0.0000,  0.0000,  0.0000],
        [-0.0988,  0.0177, -0.0527,  ...,  0.0000,  0.0000,  0.0000],
        ...,
        [-0.1784,  0.0301, -0.0544,  ...,  0.0000,  0.0000,  0.0000],
        [-0.0684,  0.0282, -0.0025,  ...,  0.0000,  0.0000,  0.0000],
        [-0.0888,  0.0044,  0.0044,  ...,  1.0000,  0.0000,  0.0000]])

In [55]:
graph[('user','rates','movie')].edge_index

tensor([[   0,    0,    0,  ...,  609,  609,  609],
        [   0,    2,    5,  ..., 9462, 9463, 9503]])

In [22]:
loader = tg.loader.DataLoader(dataset)

In [23]:
for batch in loader:
    print(type(batch))

<class 'abc.HeteroDataBatch'>


In [37]:
batch.keys()

['time', 'ptr', 'edge_label', 'edge_index', 'x', 'batch', 'num_nodes']

In [58]:
# Split edges into train/val/test
transform = tg.transforms.RandomLinkSplit(
    num_val=0.1,      # 10% validation edges
    num_test=0.1,     # 10% test edges
    is_undirected=True,
    edge_types=('user', 'rates', 'movie'),
    rev_edge_types=('movie', 'rev_rates', 'user'),
)

train_data, val_data, test_data = transform(graph)